# Notebook 01d · Milan, Sentinel-2 composite, GHS-BUILT-S labels

Predicts impervious surface density (%) for Milan from one Sentinel-2 composite (`COMPOSITE_METHOD`: median, stack or percentile; run once per composite), trained on
GHS-BUILT-S. All 3,500 points of the feature table (`data/sample_features/`,
written by notebook 00) are used; there is no train/test split.

**The random forest is tuned and built in Earth Engine** (`gee_models.py`), so
the map is the tuned model by construction:

1. Tuning: 40 settings x 3 block sizes (500 / 1000 / 2000 m), 5 spatial-block
   folds each, scored in Earth Engine; the lowest mean fold RMSE wins.
2. Spatial CV metrics: out-of-fold predictions at the chosen block size.
3. Final model: the chosen setting trained on all points; its record
   (`model_rf.json`: points, settings, seed, CV metrics, importances) is kept
   beside the map and rebuilds the identical forest (`gee_models.load_model`).
4. The map is exported and downloaded automatically (`gee_export.py`).

Accuracy against the ground is notebook 05 (EarthLabel); notebook 04 compares
the whole map with its training product.

---
## Cell 1 · Imports & configuration

In [30]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee

init_gee()

In [31]:
import os, pyproj
_proj = pyproj.datadir.get_data_dir()
os.environ['PROJ_LIB']  = _proj
os.environ['PROJ_DATA'] = _proj

In [32]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee
from gee_export import start_export, wait_and_download
import sample_features as sf
import gee_models as gm

init_gee()

import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings, os
import matplotlib.ticker as mticker
import rioxarray as rxr
from s2_utils import S2_BANDS, valid_mask, water_mask, build_composite
warnings.filterwarnings('ignore')
# ── Configuration ──────────────────────────────────────────────────────────────
from gee_init import get_project
GEE_PROJECT    = get_project()   # from .env
S2_YEAR        = 2018
RANDOM_STATE   = 42
N_JOBS         = -1       # all cores; with a fixed random_state RF results do not depend on n_jobs
N_FOLDS        = 5        # CV folds within training set
INNER_CV       = 5        # inner folds for HP tuning (can be changed, for example 3 or 5)
BUFFER_M       = 250      # buffer between train/test and between CV folds
SPLIT_BLOCK_M  = 1000     # block size for train/test split
BLOCK_SIZES    = [500, 1000, 2000]   # block sizes tested for HP tuning CV
TEST_FRAC      = 0.30     # target test fraction
N_ITER_RF      = 40
N_CLASSES      = 7
# ── Composite method  <-- must match the notebook-00 run being modelled ───────
# 'median' reads the original ./samples_S2 + ./outputs_S2 (the frozen baseline);
# any other method reads ./samples_S2_<method> and writes ./outputs_S2_<method>.
# The method itself is read back from the metadata JSON below and asserted
# against the composite rebuilt in Cell 2 -- notebook 00 is the source of truth.
COMPOSITE_METHOD = 'percentile'  # 'median', 'mean', 'stack', or 'percentile'
# Percentile set of the notebook-00 run to model, mirroring PERCENTILES there.
# Only used to LOCATE that run's directory: the authoritative set is read back
# from its metadata JSON below and asserted against this, so a mismatch names a
# directory whose metadata then contradicts it rather than silently modelling
# the wrong table. Ignored unless COMPOSITE_METHOD == 'percentile'.
PERCENTILES = (10, 25, 50, 75, 90)
# ── Paths ──────────────────────────────────────────────────────────────────────
# Points, labels, split and Sentinel-2 bands come from the feature table notebook
# 00 wrote; outputs go to one directory per composite.
PREDICTOR = f'S2{COMPOSITE_METHOD}'
gdf_all, _s2_meta = sf.load_features('GHSL', 'Milan', PREDICTOR)
OUTPUT_DIR     = f'./output/milan/ghsl/{COMPOSITE_METHOD}'
RF_RASTER_PATH = f'{OUTPUT_DIR}/GHSL_predicted_RF_S2_Milan.tif'
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ── Sentinel-2 predictor: rebuilt from the table's own record ─────────────────
# The same dates and composite the bands were extracted from, so the raster is
# classified on the image the model was trained on.
S2_COLLECTION  = _s2_meta['collection']
SELECTED_DATES = _s2_meta['selected_dates']
if _s2_meta.get('percentiles'):
    PERCENTILES = tuple(_s2_meta['percentiles'])
assert _s2_meta['method'] == COMPOSITE_METHOD
print(f'Modelling data/sample_features/GHSL_Milan_{PREDICTOR}  ->  {OUTPUT_DIR}')
print(f'  method={COMPOSITE_METHOD}  dates={len(SELECTED_DATES)}  '
      f'bands={len(_s2_meta["band_names"])}')
# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})
CLASS_COLORS = ['#2166ac','#74add1','#abd9e9','#fee090','#f46d43','#d73027','#a50026']
CLASS_LABELS = ['C0 (0%)','C1 (1-20%)','C2 (21-40%)','C3 (41-60%)','C4 (61-80%)','C5 (81-99%)','C6 (100%)']
MODEL_COLORS = {'RF': '#2ecc71'}
CV_LABELS    = ['Random'] + [f'{bs}m' for bs in BLOCK_SIZES]
print('Setup OK  ·  SENTINEL-2 BASELINE')
print(f'Predictor: {S2_COLLECTION} | {len(SELECTED_DATES)} date(s)')
print(f'Split block: {SPLIT_BLOCK_M}m | Buffer: {BUFFER_M}m | Target test: {TEST_FRAC*100:.0f}%')
print(f'CV folds: {N_FOLDS} outer / {INNER_CV} inner | Block sizes: {BLOCK_SIZES} m')


Modelling ./samples_S2_percentile_p10p25p50p75p90  ->  ./outputs_S2_percentile_p10p25p50p75p90_GHSL
  method=percentile  dates=30  bands=50
Setup OK  ·  SENTINEL-2 BASELINE
Predictor: COPERNICUS/S2_SR_HARMONIZED | 30 date(s)
Split block: 1000m | Buffer: 250m | Target test: 30%
CV folds: 5 outer / 5 inner | Block sizes: [500, 1000, 2000] m
Composite method: percentile (10, 25, 50, 75, 90) | 50 bands from 30 date(s)
  table : ./samples_S2_percentile_p10p25p50p75p90/sample_points_all_S2_GHSL.gpkg
  output: ./outputs_S2_percentile_p10p25p50p75p90_GHSL


---
## Cell 2 · Load GEE assets & all 3500 points

In [34]:
# ── GEE ───────────────────────────────────────────────────────────────────────
aoi      = ee.FeatureCollection(f'projects/{GEE_PROJECT}/assets/milano_aoi')
aoi_geom = aoi.geometry()

non_water_mask = water_mask(aoi_geom)

# ── Sentinel-2 composite -- built by the SAME helper as notebook 00 ───────────
# The mask/composite definition must not drift between the two notebooks: the
# holdout evaluation compares points sampled from the raster exported here
# against a model trained on the table extracted there.
# percentiles= is ignored by 'median'/'stack'. The band-name assert below is
# what actually guarantees this composite matches the extraction table.
pred_img, s2_proj, band_names = build_composite(
    S2_COLLECTION, SELECTED_DATES, aoi_geom, non_water_mask,
    method=COMPOSITE_METHOD, percentiles=PERCENTILES)

# band_names is method-dependent and is the feature-column contract between the
# two notebooks. If it disagrees with what notebook 00 recorded, the model would
# be trained on one set of columns and the raster classified with another --
# silently, and with plausible-looking metrics.
assert band_names == _s2_meta['band_names'], (
    'Band-name mismatch between this composite and the extraction table.\n'
    f'  here ({len(band_names)}): {band_names[:6]} ...\n'
    f'  meta ({len(_s2_meta["band_names"])}): {_s2_meta["band_names"][:6]} ...\n'
    'Re-run notebook 00 -- s2_utils.build_composite has changed since.')

print(f'S2 bands: {len(band_names)}  {band_names}')
print(f'Dates ({len(SELECTED_DATES)}): {SELECTED_DATES}')

# ── Points, labels, split and bands: the feature table loaded in Cell 1 ──────

gdf_utm = gdf_all.to_crs('EPSG:32632')
coords_all = np.column_stack([gdf_utm.geometry.x, gdf_utm.geometry.y])

X_all = gdf_all[band_names].values.astype(np.float32)
y_all = gdf_all['GHSL'].values.astype(np.float32)

print(f'Total points: {len(gdf_all)}')
print(f'X shape: {X_all.shape} | y range: {y_all.min():.0f}-{y_all.max():.0f}%')
print(f'Class distribution:')
for c in range(N_CLASSES):
    n = int((gdf_all['GHSL_class']==c).sum())
    print(f'  {CLASS_LABELS[c]}: {n}')


S2 bands: 50  ['B2_p10', 'B2_p25', 'B2_p50', 'B2_p75', 'B2_p90', 'B3_p10', 'B3_p25', 'B3_p50', 'B3_p75', 'B3_p90', 'B4_p10', 'B4_p25', 'B4_p50', 'B4_p75', 'B4_p90', 'B5_p10', 'B5_p25', 'B5_p50', 'B5_p75', 'B5_p90', 'B6_p10', 'B6_p25', 'B6_p50', 'B6_p75', 'B6_p90', 'B7_p10', 'B7_p25', 'B7_p50', 'B7_p75', 'B7_p90', 'B8_p10', 'B8_p25', 'B8_p50', 'B8_p75', 'B8_p90', 'B8A_p10', 'B8A_p25', 'B8A_p50', 'B8A_p75', 'B8A_p90', 'B11_p10', 'B11_p25', 'B11_p50', 'B11_p75', 'B11_p90', 'B12_p10', 'B12_p25', 'B12_p50', 'B12_p75', 'B12_p90']
Dates (30): ['2018-03-23', '2018-04-17', '2018-04-22', '2018-06-01', '2018-06-16', '2018-06-21', '2018-07-01', '2018-07-06', '2018-07-26', '2018-07-31', '2018-08-05', '2018-08-10', '2018-08-15', '2018-08-20', '2018-08-30', '2018-09-04', '2018-09-09', '2018-09-14', '2018-09-19', '2018-09-24', '2018-09-29', '2018-10-04', '2018-10-09', '2018-10-14', '2018-10-19', '2018-10-24', '2018-11-18', '2018-11-28', '2018-12-03', '2018-12-08']
Total points: 3500
X shape: (3500, 50

---
## Cell 3 · Tune in Earth Engine

Every setting is trained and scored on the 5 spatial-block folds of each block
size by Earth Engine (one request per setting and block, 8 in parallel). The
full table is saved; the chosen setting is the lowest mean fold RMSE.

In [ ]:
LABEL, SOURCE, PREDICTOR = 'GHSL', 'GHSL', PREDICTOR
MODEL_PATH = f'{OUTPUT_DIR}/model_rf.json'

fc = gm.points_fc(gdf_all, band_names, LABEL, gm.make_folds(coords_all))
tune_rows, best = gm.tune(fc, band_names, LABEL)
pd.DataFrame(tune_rows).to_csv(f'{OUTPUT_DIR}/hyperparameter_tuning.csv', index=False)
print(f'Saved {OUTPUT_DIR}/hyperparameter_tuning.csv ({len(tune_rows)} rows)')

---
## Cell 4 · Spatial CV metrics (out-of-fold)

Each point predicted by the chosen setting trained without its spatial block
(chosen block size). RMSE, MAE, R² and bias (observed minus predicted).

In [ ]:
oof_pred, cv = gm.oof_predictions(fc, band_names, LABEL, best)
pd.DataFrame({'pid': gdf_all['pid'].values, LABEL: y_all, 'pred_oof': oof_pred}).to_csv(
    f'{OUTPUT_DIR}/oof_predictions.csv', index=False)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_all, oof_pred, s=6, alpha=0.35, linewidths=0)
ax.plot([0, 100], [0, 100], 'k--', lw=1)
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect('equal')
ax.set_xlabel(f'{LABEL} (%)'); ax.set_ylabel('Out-of-fold prediction (%)')
ax.text(0.04, 0.96, f"RMSE {cv['rmse']:.2f}\nMAE {cv['mae']:.2f}\nR² {cv['r2']:.3f}\n"
        f"bias {cv['bias']:+.2f}\nblock {cv['block']} m", transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='grey'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_oof_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 5 · Final model and map

The chosen setting trained on all points, its record saved (`model_rf.json`),
and the full-resolution map exported and downloaded to `RF_RASTER_PATH`.

In [ ]:
gee_rf = gm.train_final(fc, band_names, LABEL, best)
record = gm.save_model(gee_rf, gm.model_record(best, tune_rows, cv, SOURCE, 'Milan', PREDICTOR,
                                               band_names, LABEL, len(gdf_all)), MODEL_PATH)

prediction_rf = (pred_img.select(band_names).classify(gee_rf).rename('GHSL_predicted')
                 .clip(aoi_geom).updateMask(non_water_mask))
wait_and_download([start_export(prediction_rf, RF_RASTER_PATH,
                                region=aoi_geom, crs='EPSG:32632')])

---
## Cell 21 · Figure E — Raster comparison map

In [ ]:
# Figure E -- Raster comparison
DISPLAY_SCALE  = 2
GHSL_OBS_PATH  = 'data/GHSL_2018_Milan_UTM32N.tif'
GHSL_PRED_PATH = RF_RASTER_PATH

ghsl_obs  = rxr.open_rasterio(GHSL_OBS_PATH,  masked=False).squeeze('band', drop=True)[::DISPLAY_SCALE, ::DISPLAY_SCALE]
ghsl_pred = rxr.open_rasterio(GHSL_PRED_PATH, masked=False).squeeze('band', drop=True)[::DISPLAY_SCALE, ::DISPLAY_SCALE]
if ghsl_pred.rio.crs != ghsl_obs.rio.crs or ghsl_pred.shape != ghsl_obs.shape:
    ghsl_pred = ghsl_pred.rio.reproject_match(ghsl_obs)
ghsl_diff = ghsl_pred - ghsl_obs
ghsl_cmap = ListedColormap(['#1a9641','#a6d96a','#ffffbf','#fdae61','#d7191c'])

def imshow_aspect(ax, da, **kw):
    b  = da.rio.bounds()
    im = ax.imshow(da.values, extent=(b[0], b[2], b[1], b[3]), **kw)
    ax.set_aspect('equal')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
    ax.tick_params(axis='both', labelsize=8)
    ax.set_xlabel('Easting [m]', fontsize=9); ax.set_ylabel('Northing [m]', fontsize=9)
    return im

fig, axes_e = plt.subplots(2, 2, figsize=(15, 12), gridspec_kw={'wspace': 0.25, 'hspace': 0.04})
im1 = imshow_aspect(axes_e[0,0], ghsl_obs,  cmap=ghsl_cmap, vmin=0, vmax=100)
axes_e[0,0].set_title('Observed GHSL (2018)', fontweight='bold', fontsize=12)
im2 = imshow_aspect(axes_e[0,1], ghsl_pred, cmap=ghsl_cmap, vmin=0, vmax=100)
axes_e[0,1].set_title('Predicted GHSL (Sentinel-2)', fontweight='bold', fontsize=12)
im3 = imshow_aspect(axes_e[1,0], ghsl_diff, cmap='RdBu_r', vmin=-30, vmax=30)
axes_e[1,0].set_title('Difference (Predicted - Observed)', fontweight='bold', fontsize=12)
axes_e[1,1].axis('off')

fig.colorbar(im1, ax=axes_e[0,:], orientation='vertical', fraction=0.025, pad=0.02, label='GHSL (%)')
fig.colorbar(im3, ax=axes_e[1,:], orientation='vertical', fraction=0.025, pad=0.02, label='Pred - Obs (%)')
plt.savefig(f'{OUTPUT_DIR}/figE_raster_comparison_1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure E saved.')

---
## Cell 7 · Summary

In [ ]:
print('=' * 70)
print(f'  {SOURCE} Milan {PREDICTOR}: random forest tuned in Earth Engine, all {len(gdf_all)} points')
print('=' * 70)
print(f"  chosen: block {best['block']} m, {gm.ee_kwargs(best)}")
print(f"  spatial CV (out-of-fold): RMSE {cv['rmse']:.2f}  MAE {cv['mae']:.2f}  "
      f"R2 {cv['r2']:.3f}  bias {cv['bias']:+.2f}")
print(f'  model record: {MODEL_PATH}')
print(f'  map:          {RF_RASTER_PATH}')